In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM-135M"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

text = "What is the capital of France?"
inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=20, use_cache=False)
print(tokenizer.decode(outputs, skip_special_tokens=True)[0])

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

What is the capital of France?
Paris, the capital of France, is the largest city in the country and the second-largest


Look at this [graph](https://www.desmos.com/calculator/awpzcx2l21) to see number of tokens to complete a full rotation as a function of dial index. As you can see, it's exponential and this is because tokens closer in position tend to be more significantly related than further ones in kind of a logarithmic fashion, if that makes sense.

In [2]:
from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding
import torch

# D = head dim
# b = base frequency term (actually wavelength, but called "frequency term" because it influences frequency)
def forward(
    self,
    x: torch.Tensor,  # for reading dtype and device
    position_ids: torch.Tensor,  # (B, S)
) -> tuple[torch.Tensor, torch.Tensor]:
    D = self.config.head_dim
    b = getattr(self.config, "rope_theta", 10000)
    dtype = x.dtype
    device = x.device

    dials = torch.arange(0, D // 2, dtype=dtype, device=device)  # (D // 2)
    inv_freq = 1 / b ** (dials / (D // 2))  # (D // 2)
    inv_freq = torch.cat([inv_freq, inv_freq])  # (D)

    position_ids = position_ids.unsqueeze(-1)  # (B, S, 1)
    inv_freq = inv_freq.unsqueeze(0)  # (1, D)
    angles = position_ids.to(dtype) @ inv_freq  # (B, S, D)

    cos = torch.cos(angles)  # (B, S, D) 
    sin = torch.sin(angles)  # (B, S, D) 
    return cos, sin

LlamaRotaryEmbedding.forward = forward

outputs = model.generate(**inputs, max_new_tokens=20, use_cache=False)
print(tokenizer.decode(outputs, skip_special_tokens=True)[0])

What is the capital of France?
Paris, the capital and largest city of France, is the seat of the French government and the


Woah it's faster. I think my rope_theta is different from the default

In [3]:
model.config.rope_theta = 5000
outputs = model.generate(**inputs, max_new_tokens=20, use_cache=False)
print(tokenizer.decode(outputs, skip_special_tokens=True)[0])

What is the capital of France?
Paris, the capital of France, is the largest city in the country and the second-largest


I think higher rope_theta makes the model talk faster because the angle between tokens increases so it feels like more time passing between each token.